# Caracterização dos volumes CCTA: ImageCAS, OrCaScore e MM-WHS

A comparação quantitativa usa somente imagens CCTA com contraste:

- **OrCaScore:** arquivos `*CTAI.mhd`; os pares `*CTI.mhd` sem contraste ficam fora da comparação entre bancos, mas entram em uma análise pareada específica e na inspeção visual.
- **MM-WHS:** arquivos `ct_*_image.nii.gz`; volumes de MRI e máscaras `*_label` são excluídos.
- **ImageCAS:** arquivos `*.img.nii.gz`; as máscaras `*.label.nii.gz` não entram no inventário.

A análise descreve inventário, geometria, cobertura física, intensidades HU e aparência das imagens. Os volumes seguem o padrão de `preprocessing_visualization.ipynb`: layout nativo `(x, y, z)`, sem canonicalização automática, e vistas axiais em `volume[:, :, z]` com `origin='upper'`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").is_file()
)
sys.path.insert(0, str(REPO_ROOT / "src"))

from utils.project.datasets.ccta import (  # noqa: E402
    discover_ccta_volumes,
    discover_orcascore_acquisitions,
    load_ccta_volume,
    load_mhd_volume_xyz,
    select_representative_exams,
    summarize_ccta_inventory,
)
from utils.project.dataframe import require_series_column  # noqa: E402
from utils.project.runtime.notebook_env import resolve_existing_path  # noqa: E402
from utils.project.runtime.notebook_env import resolve_imagecas_base_path  # noqa: E402

pd.set_option("display.max_columns", 30)
plt.style.use("seaborn-v0_8-whitegrid")

HU_EXAMS_PER_DATASET = 30
HU_VOXELS_PER_EXAM = 50_000
HU_RANDOM_SEED = 42

In [ ]:
ORCASCORE_PATH = resolve_existing_path(
    "ORCASCORE_BASE_PATH",
    [Path("/run/media/matheus/HD/DatasetsCCTA/Orca_Score_Calcium")],
    "OrCaScore",
)
MMWHS_PATH = resolve_existing_path(
    "MMWHS_BASE_PATH",
    [Path("/run/media/matheus/HD/DatasetsCCTA/MM-WHS-2017-Dataset")],
    "MM-WHS 2017",
)
IMAGECAS_PATH = resolve_imagecas_base_path()

print(f"OrCaScore: {ORCASCORE_PATH}")
print(f"MM-WHS:    {MMWHS_PATH}")
print(f"ImageCAS:  {IMAGECAS_PATH}")

## 1. Inventário CCTA

O inventário inclui somente os padrões definidos para CCTA; CT sem contraste, MRI e rótulos ficam fora das estatísticas.

In [ ]:
inventory = discover_ccta_volumes(ORCASCORE_PATH, MMWHS_PATH, IMAGECAS_PATH)
summary = summarize_ccta_inventory(inventory)

display(summary)

In [ ]:
display(pd.crosstab(inventory["dataset"], inventory["subset"], margins=True))

## 2. Geometria e resolução espacial

Os bancos diferem na amostragem e na extensão física. ImageCAS concentra-se em 275 fatias e `0,5 mm` no eixo z; MM-WHS usa principalmente `0,625 mm`; OrCaScore apresenta a maior variação de espaçamento e quantidade de fatias. O FOV no plano explica parte da diferença aparente de zoom.

In [ ]:
colors = {
    "ImageCAS": "#440154",
    "OrCaScore": "#31688e",
    "MM-WHS": "#35b779",
}
dataset_order = list(colors)
box_metrics = [
    ("slice_count", "Número de fatias"),
    ("spacing_x_mm", "Espaçamento no plano (mm)"),
    ("coverage_z_mm", "Cobertura axial (mm)"),
    ("fov_x_mm", "FOV no plano (mm)"),
    ("voxel_volume_mm3", "Volume do voxel (mm³)"),
]
fig, axes = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
box_axes = [axes[0, 0], axes[0, 2], *axes[1]]
for ax, (column, label) in zip(box_axes, box_metrics, strict=True):
    groups = [inventory.loc[inventory["dataset"].eq(name), column] for name in colors]
    boxes = ax.boxplot(groups, tick_labels=list(colors), patch_artist=True)
    for patch, color in zip(boxes["boxes"], colors.values()):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    ax.set_ylabel(label)

spacing_ax = axes[0, 1]
spacing_counts = (
    inventory.assign(spacing_z_rounded=inventory["spacing_z_mm"].round(3))
    .groupby(["dataset", "spacing_z_rounded"])
    .size()
)
max_frequency = int(spacing_counts.to_numpy().max())
for position, dataset in enumerate(dataset_order, start=1):
    counts = spacing_counts.loc[dataset]
    spacing_ax.scatter(
        np.full(len(counts), position),
        counts.index,
        s=45 + 255 * counts.to_numpy() / max_frequency,
        color=colors[dataset],
        alpha=0.8,
        edgecolor="black",
    )
    for spacing, count in counts.items():
        spacing_ax.annotate(
            f"n={count}",
            (position, spacing),
            xytext=(8, 0),
            textcoords="offset points",
            va="center",
            fontsize=8,
        )
spacing_ax.set(
    xticks=range(1, len(dataset_order) + 1),
    xticklabels=dataset_order,
    ylabel="Espaçamento axial (mm)",
    title="Valores discretos; tamanho indica frequência",
)
spacing_ax.set_xlim(0.5, len(dataset_order) + 0.65)
fig.suptitle("Características geométricas dos volumes CCTA")
plt.show()

### OrCaScore: aquisições com e sem contraste

Os exames `CTAI` e `CTI` são comparados por paciente. A quantidade de fatias é analisada junto ao espaçamento e à cobertura axial, pois esses atributos descrevem aspectos diferentes da amostragem do volume.

In [ ]:
orca_acquisitions = discover_orcascore_acquisitions(ORCASCORE_PATH)
acquisition_labels = {
    "contrast": "Com contraste (CTAI)",
    "noncontrast": "Sem contraste (CTI)",
}
acquisition_order = list(acquisition_labels)
orca_acquisitions["acquisition_label"] = require_series_column(
    orca_acquisitions, "acquisition"
).map(lambda acquisition: acquisition_labels.get(str(acquisition)))

orca_acquisition_summary = (
    orca_acquisitions.groupby("acquisition_label", sort=False)
    .agg(
        exames=("exam_id", "size"),
        fatias_media=("slice_count", "mean"),
        fatias_dp=("slice_count", "std"),
        fatias_mediana=("slice_count", "median"),
        fatias_min=("slice_count", "min"),
        fatias_max=("slice_count", "max"),
        espacamento_z_mediano_mm=("spacing_z_mm", "median"),
        cobertura_z_media_mm=("coverage_z_mm", "mean"),
        cobertura_z_mediana_mm=("coverage_z_mm", "median"),
    )
    .round(1)
)
display(orca_acquisition_summary)

metrics = [
    ("slice_count", "Número de fatias"),
    ("spacing_z_mm", "Espaçamento axial (mm)"),
    ("coverage_z_mm", "Cobertura axial (mm)"),
]
acquisition_colors = ["#31688e", "#d95f02"]
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
for ax, (column, ylabel) in zip(axes, metrics, strict=True):
    groups = [
        orca_acquisitions.loc[orca_acquisitions["acquisition"].eq(acquisition), column]
        for acquisition in acquisition_order
    ]
    boxes = ax.boxplot(
        groups,
        tick_labels=["Com contraste", "Sem contraste"],
        patch_artist=True,
        showfliers=False,
    )
    for patch, color in zip(boxes["boxes"], acquisition_colors, strict=True):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    means = [values.mean() for values in groups]
    ax.scatter([1, 2], means, marker="D", color="black", s=28, zorder=3, label="Média")
    ax.set_ylabel(ylabel)
axes[0].legend(loc="upper right", frameon=True)
fig.suptitle("OrCaScore: geometria das aquisições pareadas")
plt.show()

paired_slices = orca_acquisitions.pivot(
    index=["subset", "exam_id"], columns="acquisition", values="slice_count"
).dropna()
slice_difference = paired_slices["contrast"] - paired_slices["noncontrast"]
print(
    f"Pares completos: {len(paired_slices)} | "
    f"contraste com mais fatias: {(slice_difference > 0).sum()} | "
    f"diferença média: {slice_difference.mean():.1f} fatias"
)

**Análise.** Todos os pares possuem mais fatias na aquisição com contraste. Entretanto, o CT sem contraste usa fatias substancialmente mais espessas e mantém cobertura axial próxima à do CCTA. Portanto, a diferença na contagem representa principalmente uma amostragem axial mais fina no exame com contraste, com impacto no detalhamento 3D, no uso de memória e no tempo de processamento, e não uma diferença proporcional na extensão anatômica coberta.

## 3. Exemplos de imagens

Cada banco é representado pelo exame mais próximo da mediana da quantidade de fatias. As vistas mostram o início, o centro e o final dos volumes na mesma janela `[-200, 800] HU`.

In [ ]:
representatives = select_representative_exams(inventory)
VOLUME_CACHE = {}


def get_volume(record):
    key = str(record["path"])
    if key not in VOLUME_CACHE:
        VOLUME_CACHE[key] = load_ccta_volume(record)
    return VOLUME_CACHE[key]


def axial_views(volume):
    middle = volume.shape[2] // 2
    return [
        ("MIP axial", np.max(volume, axis=2), -200, 1200),
        ("Primeira fatia (0)", volume[:, :, 0], -200, 800),
        (f"Fatia média ({middle})", volume[:, :, middle], -200, 800),
        (f"Última fatia ({volume.shape[2] - 1})", volume[:, :, -1], -200, 800),
    ]


def plot_axial_overview(rows, title):
    fig, axes = plt.subplots(
        len(rows),
        4,
        figsize=(15, 4 * len(rows)),
        squeeze=False,
        constrained_layout=True,
    )
    for row_index, (row_label, volume) in enumerate(rows):
        for ax, (view_title, image, vmin, vmax) in zip(
            axes[row_index], axial_views(volume)
        ):
            ax.imshow(image, cmap="gray", vmin=vmin, vmax=vmax, origin="upper")
            ax.set_title(view_title)
            ax.axis("off")
        axes[row_index, 0].text(
            0.02,
            0.98,
            f"{row_label}\n{volume.shape[2]} fatias",
            transform=axes[row_index, 0].transAxes,
            va="top",
            color="white",
            fontsize=10,
            bbox={"facecolor": "black", "alpha": 0.65, "pad": 4},
        )
    fig.suptitle(title, fontsize=14)
    plt.show()

### OrCaScore

In [ ]:
orca_record = (
    representatives.loc[representatives["dataset"].eq("OrCaScore")]
    .reset_index(drop=True)
    .iloc[1]
)
orca_ccta = get_volume(orca_record)
orca_noncontrast_path = Path(orca_record["path"]).with_name(
    Path(orca_record["path"]).name.replace("CTAI.mhd", "CTI.mhd")
)
if orca_noncontrast_path.is_file():
    orca_noncontrast = load_mhd_volume_xyz(orca_noncontrast_path)
    plot_axial_overview(
        [
            ("CCTA com contraste (CTAI)", orca_ccta),
            ("CT sem contraste (CTI)", orca_noncontrast),
        ],
        title=f"OrCaScore · par do paciente {orca_record['exam_id']}",
    )
else:
    print(f"Par CTI ausente; comparação não exibida: {orca_noncontrast_path}")

### MM-WHS

In [ ]:
mmwhs_record = (
    representatives.loc[representatives["dataset"].eq("MM-WHS")]
    .reset_index(drop=True)
    .iloc[1]
)
mmwhs_volume = get_volume(mmwhs_record)
plot_axial_overview(
    [("CCTA", mmwhs_volume)],
    title=f"MM-WHS · {mmwhs_record['exam_id']}",
)

### ImageCAS

In [ ]:
imagecas_record = (
    representatives.loc[representatives["dataset"].eq("ImageCAS")]
    .reset_index(drop=True)
    .iloc[1]
)
imagecas_volume = get_volume(imagecas_record)
plot_axial_overview(
    [("CCTA de referência", imagecas_volume)],
    title=f"ImageCAS · exame {imagecas_record['exam_id']}",
)

### Comparação lado a lado

As colunas representam os bancos e as linhas mostram a primeira, a média e a última fatia do exame mediano de cada banco. A janela de visualização é a mesma nos nove painéis.

In [ ]:
dataset_order = ["ImageCAS", "OrCaScore", "MM-WHS"]
comparison_cases = {
    dataset: representatives.loc[representatives["dataset"].eq(dataset)]
    .reset_index(drop=True)
    .iloc[1]
    for dataset in dataset_order
}
slice_labels = ["Primeira fatia", "Fatia média", "Última fatia"]
fig, axes = plt.subplots(3, 3, figsize=(13.5, 11.5), constrained_layout=True)
for col, dataset in enumerate(dataset_order):
    record = comparison_cases[dataset]
    volume = get_volume(record)
    slice_indices = [0, volume.shape[2] // 2, volume.shape[2] - 1]
    for row, (slice_label, slice_index) in enumerate(
        zip(slice_labels, slice_indices, strict=True)
    ):
        axes[row, col].imshow(
            volume[:, :, slice_index],
            cmap="gray",
            vmin=-200,
            vmax=800,
            origin="upper",
        )
        axes[row, col].set_title(
            f"{dataset} · {record['exam_id']}\n"
            f"{slice_label} ({slice_index}/{volume.shape[2] - 1})"
        )
        axes[row, col].axis("off")
fig.suptitle("CCTA: primeira, média e última fatia por banco", fontsize=15)
plt.show()

### Efeito do campo de visão

Nos volumes brutos, o FOV maior do OrCaScore e do MM-WHS reduz visualmente as estruturas cardíacas. O recorte central comum de aproximadamente `193 mm` elimina essa diferença de enquadramento sem interpolar ou modificar os arquivos.

In [ ]:
def center_crop_to_fov(volume, spacing_x, spacing_y, target_fov_mm):
    size_x = min(volume.shape[0], round(target_fov_mm / spacing_x))
    size_y = min(volume.shape[1], round(target_fov_mm / spacing_y))
    start_x = (volume.shape[0] - size_x) // 2
    start_y = (volume.shape[1] - size_y) // 2
    return volume[start_x : start_x + size_x, start_y : start_y + size_y, :]


target_fov_mm = min(
    min(float(record["fov_x_mm"]), float(record["fov_y_mm"]))
    for record in comparison_cases.values()
)
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.5), constrained_layout=True)
for ax, dataset in zip(axes, dataset_order, strict=True):
    record = comparison_cases[dataset]
    volume = get_volume(record)
    cropped = center_crop_to_fov(
        volume,
        float(record["spacing_x_mm"]),
        float(record["spacing_y_mm"]),
        target_fov_mm,
    )
    middle = cropped.shape[2] // 2
    ax.imshow(
        cropped[:, :, middle],
        cmap="gray",
        vmin=-200,
        vmax=800,
        origin="upper",
    )
    actual_fov_x = cropped.shape[0] * float(record["spacing_x_mm"])
    actual_fov_y = cropped.shape[1] * float(record["spacing_y_mm"])
    ax.set_title(
        f"{dataset} · {record['exam_id']}\n"
        f"FOV central {actual_fov_x:.1f} × {actual_fov_y:.1f} mm"
    )
    ax.axis("off")
fig.suptitle(
    f"Fatia média com FOV central comum de aproximadamente {target_fov_mm:.1f} mm",
    fontsize=14,
)
plt.show()

## 4. Orientação anatômica e referência ImageCAS

Os cabeçalhos classificam todos os ImageCAS como `LAS`, 58 MM-WHS como `LAS`, dois MM-WHS como `RAS` e todos os OrCaScore como `RAI`. As figuras preservam a ordem nativa dos voxels e mostram o OrCaScore invertido horizontalmente em relação à apresentação do ImageCAS.

In [ ]:
orientation_counts = inventory.groupby(["dataset", "reported_orientation"]).size()
orientation_table = pd.DataFrame({"exams": orientation_counts}).reset_index()
display(orientation_table)

### Análise da orientação visual

A comparação visual mostra o OrCaScore invertido horizontalmente em relação ao ImageCAS. No layout usado pelo notebook, `(x, y, z)`, a correção corresponde a `volume[:, ::-1, :]`: somente o eixo 1 é invertido. A transformação mantém o eixo 0, a ordem das fatias e os valores dos voxels. Ela padroniza a apresentação das imagens, mas não substitui uma reorientação física baseada na matriz de direção.

### Correção da orientação visual

Os três pares mostram a mesma fatia antes e depois da inversão horizontal, aplicada no eixo 1. Os eixos 0 e z permanecem inalterados.

In [ ]:
def correct_orcascore_visual_orientation(record, volume):
    """Flip the OrCaScore second axis for consistent visual display."""
    if record["dataset"] == "OrCaScore":
        return volume[:, ::-1, :]
    return volume


orca_examples = representatives.loc[
    representatives["dataset"].eq("OrCaScore")
].reset_index(drop=True)
fig, axes = plt.subplots(3, 2, figsize=(9, 12), constrained_layout=True)
for row, (_, record) in enumerate(orca_examples.iterrows()):
    original = get_volume(record)
    corrected = correct_orcascore_visual_orientation(record, original)
    middle = original.shape[2] // 2
    for ax, image, stage in zip(
        axes[row],
        [original[:, :, middle], corrected[:, :, middle]],
        ["Antes: orientação nativa", "Depois: orientação corrigida"],
        strict=True,
    ):
        ax.imshow(image, cmap="gray", vmin=-200, vmax=800, origin="upper")
        ax.set_title(
            f"{stage}\n{record['exam_id']} · fatia {middle}/{original.shape[2] - 1}"
        )
        ax.axis("off")
fig.suptitle("OrCaScore: alinhamento visual com o ImageCAS", fontsize=14)
plt.show()

## 5. Distribuição de intensidades

Para equilibrar os bancos e limitar memória e tempo, o gráfico sorteia `HU_EXAMS_PER_DATASET` exames de cada banco e até `HU_VOXELS_PER_EXAM` voxels por exame. A semente fixa torna a amostra reproduzível. A faixa exibida (`-1000` a `1500 HU`) cobre ar, pulmão, gordura, tecidos moles, contraste e cálcio; valores extremos permanecem nos volumes originais.

In [ ]:
rng = np.random.default_rng(HU_RANDOM_SEED)
hu_exams = pd.concat(
    [
        group.sample(
            n=min(HU_EXAMS_PER_DATASET, len(group)),
            random_state=int(rng.integers(0, 2**32 - 1)),
        )
        for _, group in inventory.groupby("dataset", sort=False)
    ],
    ignore_index=True,
)

samples = []
for _, record in hu_exams.iterrows():
    # O carregamento direto evita manter todos os volumes sorteados no cache.
    values = np.asarray(load_ccta_volume(record), dtype=np.float32).reshape(-1)
    if values.size > HU_VOXELS_PER_EXAM:
        values = values[rng.choice(values.size, HU_VOXELS_PER_EXAM, replace=False)]
    values = values[np.isfinite(values)]
    values = values[(values >= -1000) & (values <= 1500)]
    samples.append(
        pd.DataFrame({"dataset": record["dataset"], "hu": values.astype(float)})
    )
hu_sample = pd.concat(samples, ignore_index=True)
hu_sample["dataset"] = hu_sample["dataset"].astype("category")

fig, ax = plt.subplots(figsize=(10, 4.5))
bins = np.linspace(-1000, 1500, 251).tolist()
for dataset, color in colors.items():
    values = hu_sample.loc[hu_sample["dataset"].eq(dataset), "hu"]
    ax.hist(
        values,
        bins=bins,
        density=True,
        histtype="step",
        linewidth=1.5,
        color=color,
        label=dataset,
    )
ax.set(xlabel="Intensidade (HU)", ylabel="Densidade", title="Amostra de intensidades")
ax.legend()
plt.show()

hu_summary = pd.DataFrame(
    hu_sample.groupby("dataset", observed=True)["hu"].agg(
        ["count", "mean", "std", "median", "min", "max"]
    )
).rename(columns={"count": "voxels"})
exam_counts = hu_exams.groupby("dataset").size()
hu_summary.insert(0, "exams", exam_counts.reindex(hu_summary.index))
display(hu_summary.round(1))

## 6. Síntese dos resultados

- **Geometria:** ImageCAS tem FOV mediano próximo de `179 mm`, contra aproximadamente `220 mm` no OrCaScore e `228 mm` no MM-WHS. O maior FOV explica o aspecto mais afastado desses bancos.
- **Amostragem:** ImageCAS usa `0,5 mm` no eixo z; MM-WHS concentra-se em `0,625 mm`. No OrCaScore, o CCTA possui muito mais fatias que o CT sem contraste porque sua amostragem axial é mais fina, embora ambos cubram uma extensão física semelhante.
- **Orientação:** ImageCAS é uniformemente `LAS`, MM-WHS contém `LAS` e `RAS`, e OrCaScore reporta `RAI`. Na visualização nativa, a inversão do eixo 1 alinha o OrCaScore ao ImageCAS sem alterar os eixos 0 e z.
- **Domínio:** resolução, FOV e distribuição HU diferem entre as coortes, além das diferenças de orientação.
- **Rótulos:** ImageCAS fornece máscaras coronárias; MM-WHS, estruturas cardíacas; e as referências do OrCaScore pertencem ao CT sem contraste. As estatísticas deste notebook usam somente CCTA.